# 3. Markov Decision Processes (MDP)

Bu notebook, Sutton & Barto kitabının 3. bölümünü kapsar.

## İçindekiler
1. Markov Property
2. MDP Formülasyonu
3. Value Functions
4. Bellman Equations
5. Optimal Policies

## 3.1 Markov Property

Bir state **Markov** özelliğine sahipse, gelecek sadece şimdiki state'e bağlıdır, geçmişe değil:

$$P[S_{t+1} | S_t] = P[S_{t+1} | S_1, S_2, ..., S_t]$$

"Gelecek, geçmişten bağımsızdır, şimdi verildiğinde."

Bu özellik RL'yi tractable (çözülebilir) yapar.

## 3.2 MDP Formülasyonu

Bir MDP şu tuple ile tanımlanır: $(S, A, P, R, \gamma)$

- $S$: State uzayı (sonlu veya sonsuz)
- $A$: Action uzayı
- $P(s'|s,a)$: Transition probability (dinamikler)
- $R(s,a,s')$: Reward function
- $\gamma \in [0,1]$: Discount factor

### Dynamics Function

$$p(s', r | s, a) = P[S_{t+1}=s', R_{t+1}=r | S_t=s, A_t=a]$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, Dict, List

class GridWorldMDP:
    """
    Basit bir Grid World MDP.
    Agent 4 yöne hareket edebilir: yukarı, sağ, aşağı, sol
    """
    
    def __init__(self, rows=4, cols=4, terminal_states=None, rewards=None):
        self.rows = rows
        self.cols = cols
        self.n_states = rows * cols
        self.n_actions = 4  # 0:up, 1:right, 2:down, 3:left
        
        # Terminal states (default: köşeler)
        self.terminal_states = terminal_states or [0, self.n_states - 1]
        
        # Default rewards: her adım -1 (terminal hariç)
        self.rewards = rewards or {s: -1 for s in range(self.n_states)}
        for ts in self.terminal_states:
            self.rewards[ts] = 0
        
        # Action movements
        self.action_effects = {
            0: (-1, 0),   # up
            1: (0, 1),    # right
            2: (1, 0),    # down
            3: (0, -1)    # left
        }
        
        self.action_names = ['↑', '→', '↓', '←']
    
    def state_to_pos(self, state: int) -> Tuple[int, int]:
        return state // self.cols, state % self.cols
    
    def pos_to_state(self, row: int, col: int) -> int:
        return row * self.cols + col
    
    def get_next_state(self, state: int, action: int) -> int:
        """Deterministic transition."""
        if state in self.terminal_states:
            return state
        
        row, col = self.state_to_pos(state)
        d_row, d_col = self.action_effects[action]
        
        new_row = max(0, min(self.rows - 1, row + d_row))
        new_col = max(0, min(self.cols - 1, col + d_col))
        
        return self.pos_to_state(new_row, new_col)
    
    def get_transition_prob(self, state: int, action: int, next_state: int) -> float:
        """P(s'|s, a) - Deterministic case."""
        expected_next = self.get_next_state(state, action)
        return 1.0 if next_state == expected_next else 0.0
    
    def get_reward(self, state: int, action: int, next_state: int) -> float:
        """R(s, a, s')."""
        if state in self.terminal_states:
            return 0
        return self.rewards.get(next_state, -1)
    
    def is_terminal(self, state: int) -> bool:
        return state in self.terminal_states

# Test
mdp = GridWorldMDP(rows=4, cols=4)
print(f"States: {mdp.n_states}")
print(f"Actions: {mdp.n_actions}")
print(f"Terminal states: {mdp.terminal_states}")

In [ ]:
def visualize_grid(mdp, values=None, policy=None, title="Grid World"):
    """Grid world'ü görselleştir."""
    fig, ax = plt.subplots(figsize=(8, 8))
    
    for state in range(mdp.n_states):
        row, col = mdp.state_to_pos(state)
        
        # Cell color
        if state in mdp.terminal_states:
            color = 'lightgreen'
        else:
            color = 'white'
        
        rect = plt.Rectangle((col, mdp.rows - 1 - row), 1, 1, 
                              facecolor=color, edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        
        # Value
        if values is not None:
            ax.text(col + 0.5, mdp.rows - row - 0.3, f'{values[state]:.1f}',
                   ha='center', va='center', fontsize=12)
        
        # Policy (arrows)
        if policy is not None and state not in mdp.terminal_states:
            action = policy[state]
            ax.text(col + 0.5, mdp.rows - row - 0.7, mdp.action_names[action],
                   ha='center', va='center', fontsize=16)
        
        # State number
        ax.text(col + 0.1, mdp.rows - row - 0.1, str(state),
               ha='left', va='top', fontsize=8, color='gray')
    
    ax.set_xlim(0, mdp.cols)
    ax.set_ylim(0, mdp.rows)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=14)
    plt.show()

visualize_grid(mdp, title="4x4 Grid World MDP")

## 3.3 Value Functions

### State-Value Function $V^\pi(s)$

Policy $\pi$ altında state $s$'den başlayarak beklenen return:

$$V^\pi(s) = E_\pi[G_t | S_t = s] = E_\pi\left[\sum_{k=0}^{\infty} \gamma^k R_{t+k+1} | S_t = s\right]$$

### Action-Value Function $Q^\pi(s, a)$

State $s$'de action $a$'yı alıp sonra $\pi$'yi takip etmenin değeri:

$$Q^\pi(s, a) = E_\pi[G_t | S_t = s, A_t = a]$$

### İlişki

$$V^\pi(s) = \sum_a \pi(a|s) Q^\pi(s, a)$$

## 3.4 Bellman Equations

Value function'lar **recursive** bir yapıya sahiptir.

### Bellman Expectation Equation (V için)

$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s'} p(s'|s,a) [r(s,a,s') + \gamma V^\pi(s')]$$

### Bellman Expectation Equation (Q için)

$$Q^\pi(s,a) = \sum_{s'} p(s'|s,a) \left[r(s,a,s') + \gamma \sum_{a'} \pi(a'|s') Q^\pi(s', a')\right]$$

In [ ]:
def evaluate_policy_iterative(mdp, policy, gamma=1.0, theta=1e-6):
    """
    Iterative Policy Evaluation.
    Bellman Expectation Equation'ı iteratif olarak çözer.
    """
    V = np.zeros(mdp.n_states)
    
    iteration = 0
    while True:
        delta = 0
        
        for s in range(mdp.n_states):
            if mdp.is_terminal(s):
                continue
            
            v = V[s]
            
            # Bellman expectation update
            new_v = 0
            a = policy[s]  # Deterministic policy
            
            for s_next in range(mdp.n_states):
                p = mdp.get_transition_prob(s, a, s_next)
                r = mdp.get_reward(s, a, s_next)
                new_v += p * (r + gamma * V[s_next])
            
            V[s] = new_v
            delta = max(delta, abs(v - V[s]))
        
        iteration += 1
        
        if delta < theta:
            break
    
    print(f"Policy evaluation converged in {iteration} iterations")
    return V

In [ ]:
# Random policy: her state'te rastgele action
# Burada basitlik için hep 'down' (2) diyelim
random_policy = np.random.randint(0, 4, size=mdp.n_states)
print(f"Random policy: {random_policy}")

# Uniform random policy için değerlendirme yapalım
# Her action eşit olasılıklı
def evaluate_uniform_random_policy(mdp, gamma=1.0, theta=1e-6):
    """Uniform random policy (her action 1/4 olasılık)."""
    V = np.zeros(mdp.n_states)
    
    iteration = 0
    while True:
        delta = 0
        
        for s in range(mdp.n_states):
            if mdp.is_terminal(s):
                continue
            
            v = V[s]
            new_v = 0
            
            for a in range(mdp.n_actions):
                action_prob = 1.0 / mdp.n_actions
                
                for s_next in range(mdp.n_states):
                    p = mdp.get_transition_prob(s, a, s_next)
                    r = mdp.get_reward(s, a, s_next)
                    new_v += action_prob * p * (r + gamma * V[s_next])
            
            V[s] = new_v
            delta = max(delta, abs(v - V[s]))
        
        iteration += 1
        
        if delta < theta:
            break
    
    print(f"Converged in {iteration} iterations")
    return V

V_random = evaluate_uniform_random_policy(mdp)
visualize_grid(mdp, values=V_random, title="Value Function (Random Policy)")

## 3.5 Optimal Value Functions ve Bellman Optimality

### Optimal State-Value Function

$$V^*(s) = \max_\pi V^\pi(s)$$

### Optimal Action-Value Function

$$Q^*(s, a) = \max_\pi Q^\pi(s, a)$$

### Bellman Optimality Equation (V*)

$$V^*(s) = \max_a \sum_{s'} p(s'|s,a) [r(s,a,s') + \gamma V^*(s')]$$

### Bellman Optimality Equation (Q*)

$$Q^*(s, a) = \sum_{s'} p(s'|s,a) [r(s,a,s') + \gamma \max_{a'} Q^*(s', a')]$$

In [ ]:
def value_iteration(mdp, gamma=1.0, theta=1e-6):
    """
    Value Iteration: Bellman Optimality Equation'ı iteratif çözer.
    Hem optimal V* hem de optimal policy π* döndürür.
    """
    V = np.zeros(mdp.n_states)
    
    iteration = 0
    while True:
        delta = 0
        
        for s in range(mdp.n_states):
            if mdp.is_terminal(s):
                continue
            
            v = V[s]
            
            # Find max over actions
            action_values = []
            for a in range(mdp.n_actions):
                q = 0
                for s_next in range(mdp.n_states):
                    p = mdp.get_transition_prob(s, a, s_next)
                    r = mdp.get_reward(s, a, s_next)
                    q += p * (r + gamma * V[s_next])
                action_values.append(q)
            
            V[s] = max(action_values)
            delta = max(delta, abs(v - V[s]))
        
        iteration += 1
        
        if delta < theta:
            break
    
    # Extract optimal policy
    policy = np.zeros(mdp.n_states, dtype=int)
    for s in range(mdp.n_states):
        if mdp.is_terminal(s):
            continue
        
        action_values = []
        for a in range(mdp.n_actions):
            q = 0
            for s_next in range(mdp.n_states):
                p = mdp.get_transition_prob(s, a, s_next)
                r = mdp.get_reward(s, a, s_next)
                q += p * (r + gamma * V[s_next])
            action_values.append(q)
        
        policy[s] = np.argmax(action_values)
    
    print(f"Value iteration converged in {iteration} iterations")
    return V, policy

V_star, pi_star = value_iteration(mdp)
visualize_grid(mdp, values=V_star, policy=pi_star, title="Optimal V* and π*")

In [ ]:
# Karşılaştırma: Random vs Optimal
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Random policy values
ax = axes[0]
for state in range(mdp.n_states):
    row, col = mdp.state_to_pos(state)
    color = 'lightgreen' if state in mdp.terminal_states else 'white'
    rect = plt.Rectangle((col, mdp.rows - 1 - row), 1, 1, 
                          facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(col + 0.5, mdp.rows - row - 0.5, f'{V_random[state]:.1f}',
           ha='center', va='center', fontsize=14)
ax.set_xlim(0, mdp.cols)
ax.set_ylim(0, mdp.rows)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('V(s) - Random Policy', fontsize=14)

# Optimal policy values
ax = axes[1]
for state in range(mdp.n_states):
    row, col = mdp.state_to_pos(state)
    color = 'lightgreen' if state in mdp.terminal_states else 'white'
    rect = plt.Rectangle((col, mdp.rows - 1 - row), 1, 1, 
                          facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(col + 0.5, mdp.rows - row - 0.3, f'{V_star[state]:.1f}',
           ha='center', va='center', fontsize=14)
    if state not in mdp.terminal_states:
        ax.text(col + 0.5, mdp.rows - row - 0.7, mdp.action_names[pi_star[state]],
               ha='center', va='center', fontsize=18)
ax.set_xlim(0, mdp.cols)
ax.set_ylim(0, mdp.rows)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('V*(s) and π* - Optimal', fontsize=14)

plt.tight_layout()
plt.show()

## Özet

Bu notebook'ta öğrendiklerimiz:

| Kavram | Açıklama |
|--------|----------|
| **MDP** | $(S, A, P, R, \gamma)$ tuple'ı |
| **Markov Property** | Gelecek sadece şimdiye bağlı |
| **$V^\pi(s)$** | State'in policy altındaki değeri |
| **$Q^\pi(s,a)$** | State-action çiftinin değeri |
| **Bellman Eq.** | Recursive değer ilişkisi |
| **$V^*, Q^*$** | Optimal value functions |

### Önemli Formüller

**Bellman Expectation (V)**:
$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s'} p(s'|s,a) [r + \gamma V^\pi(s')]$$

**Bellman Optimality (V)**:
$$V^*(s) = \max_a \sum_{s'} p(s'|s,a) [r + \gamma V^*(s')]$$

### Sonraki Notebook
**04 - Dynamic Programming**: Policy Iteration, Value Iteration detaylı